# 03.4 - Embeddings - NV-Embed

This notebook embeds paper abstracts using NVIDIA's [NV-Embed-v2](https://huggingface.co/nvidia/NV-Embed-v2), a high-performing open embedding model. The model is large (~7B parameters), so this notebook expects a CUDA-capable GPU and uses `DataParallel` to spread inference across multiple devices when available.

## Notebook Setup

Install NV-Embed dependencies (`einops`, `datasets`), Chroma, and `sentence_transformers`. `pynvml` is used below to inspect available CUDA devices.

In [ ]:
%pip install --upgrade --quiet einops datasets pynvml chromadb sentence_transformers

Load articles and prune ones without abstracts, since we're using the abstracts for generating the embeddings.

In [ ]:
import pandas as pd
from genscai import paths

df_modeling_papers = pd.read_json(paths.data / "modeling_papers_0.json", orient="records", lines=True)

df_modeling_papers.shape

Stage the articles so that they can easily be loaded into the vector database.

In [ ]:
documents = []
ids = []

for paper in df_modeling_papers.itertuples():
    documents.append(paper.abstract)
    ids.append(paper.id)

print(f"Number of documents: {len(documents)}")

For finding semantically related documents, we'll use Chroma (https://www.trychroma.com/), which is a lightweight vector data store. Chroma supports swappable embedding models, filtering using metadata, keyword search, and multiple distance measurements. We'll use these features for evaluating approaches to organizing papers for downstream processing (search, summarization, keyword extraction, etc.).

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="vectors_db")

Print available CUDA device info before loading the model, to confirm the GPU(s) the notebook will dispatch to.

In [ ]:
from genscai import utils

utils.print_cuda_device_info()

Bind a Chroma embedding function to NV-Embed-v2. `trust_remote_code=True` is required because NV-Embed ships custom modeling code on the Hub.

In [ ]:
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="nvidia/NV-Embed-v2",
    # device='cuda',
    trust_remote_code=True,
)

# client.delete_collection(name="articles-nv_embed_v2-embeddings")
collection = client.create_collection(name="articles2", embedding_function=ef)

Embed and add documents one at a time. The per-document loop trades throughput for visibility into which paper is currently being processed.

In [ ]:
from tqdm import tqdm

for i in tqdm(range(len(documents))):
    print(ids[i])
    print(documents[i])
    collection.add(documents=documents[i], ids=ids[i])

# collection.add(documents=documents, ids=ids)

Load NV-Embed directly with Hugging Face `transformers` and wrap each submodule in `DataParallel` to fan inference out across multiple GPUs. Wrapping submodules individually (instead of the whole model) preserves access to NV-Embed's custom forward methods.

In [ ]:
from transformers import AutoModel
from torch.nn import DataParallel

model = AutoModel.from_pretrained("nvidia/NV-Embed-v2", trust_remote_code=True, device_map="auto")

# model = DataParallel(model)
for module_key, module in model._modules.items():
    model._modules[module_key] = DataParallel(module)